In [ ]:
import pandas as pd
import numpy as np
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import classification_report
import joblib
import optuna
import os
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

# Load data
X_train = pd.read_csv('../../dataset_processed/X_train_final.csv')
y_train = pd.read_csv('../../dataset_processed/y_train_final.csv').values.ravel()


## Phase 1: Baseline Model

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ['f1_macro', 'recall_macro', 'accuracy']

base_pipeline = Pipeline([
    ('smote', SMOTE(random_state=42)),
    ('mlp', MLPClassifier(max_iter=1000, random_state=42))
])

cv_results_base = cross_validate(base_pipeline, X_train, y_train, cv=cv, scoring=scoring)
baseline_f1 = cv_results_base['test_f1_macro'].mean()

print(f"Mean CV Macro F1: {baseline_f1:.4f}")
print(f"Mean CV Macro Recall: {cv_results_base['test_recall_macro'].mean():.4f}")
print(f"Mean CV Accuracy: {cv_results_base['test_accuracy'].mean():.4f}")

base_pipeline.fit(X_train, y_train)
y_pred_base = base_pipeline.predict(X_train)
report_base = classification_report(y_train, y_pred_base)
print("\nClassification Report on Full Train Data (Baseline):\n", report_base)

os.makedirs('results', exist_ok=True)
with open('results/baseline_metrics.txt', 'w') as f:
    f.write(f"Mean CV Macro F1: {baseline_f1:.4f}\n")
    f.write(f"Mean CV Macro Recall: {cv_results_base['test_recall_macro'].mean():.4f}\n")
    f.write(f"Mean CV Accuracy: {cv_results_base['test_accuracy'].mean():.4f}\n\n")
    f.write("Classification Report on Full Train Data (Baseline):\n")
    f.write(report_base)


## Phase 2: Hyperparameter Tuning with Optuna

In [ ]:
def objective(trial):
    mlp__hidden_layer_sizes = trial.suggest_categorical('mlp__hidden_layer_sizes', [(100,), (100, 50), (64, 32), (50,)])
    mlp__activation = trial.suggest_categorical('mlp__activation', ['relu', 'tanh'])
    mlp__learning_rate_init = trial.suggest_float('mlp__learning_rate_init', 1e-4, 1e-2, log=True)
    mlp__alpha = trial.suggest_float('mlp__alpha', 1e-5, 1e-1, log=True)
    
    pipeline = Pipeline([
        ('smote', SMOTE(random_state=42)),
        ('mlp', MLPClassifier(
            max_iter=500,
            random_state=42,
            hidden_layer_sizes=mlp__hidden_layer_sizes,
            activation=mlp__activation,
            learning_rate_init=mlp__learning_rate_init,
            alpha=mlp__alpha,
            early_stopping=True,
            n_iter_no_change=10
        ))
    ])
    
    cv_results = cross_validate(pipeline, X_train, y_train, cv=cv, scoring='f1_macro')
    return cv_results['test_score'].mean()

study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=20)

print(f"Best Params: {study.best_params}")
print(f"Best Macro F1: {study.best_value:.4f}")


## Phase 3: Tuned Eval & Comparison

In [ ]:
tuned_pipeline = Pipeline([
    ('smote', SMOTE(random_state=42)),
    ('mlp', MLPClassifier(
        max_iter=500,
        random_state=42,
        hidden_layer_sizes=study.best_params['mlp__hidden_layer_sizes'],
        activation=study.best_params['mlp__activation'],
        learning_rate_init=study.best_params['mlp__learning_rate_init'],
        alpha=study.best_params['mlp__alpha'],
        early_stopping=True,
        n_iter_no_change=10
    ))
])

cv_results_tuned = cross_validate(tuned_pipeline, X_train, y_train, cv=cv, scoring=scoring)
tuned_f1 = cv_results_tuned['test_f1_macro'].mean()

print(f"Mean CV Macro F1 (Tuned): {tuned_f1:.4f}")
print(f"Mean CV Macro Recall (Tuned): {cv_results_tuned['test_recall_macro'].mean():.4f}")
print(f"Mean CV Accuracy (Tuned): {cv_results_tuned['test_accuracy'].mean():.4f}")

tuned_pipeline.fit(X_train, y_train)
y_pred_tuned = tuned_pipeline.predict(X_train)
report_tuned = classification_report(y_train, y_pred_tuned)
print("\nClassification Report on Full Train Data (Tuned):\n", report_tuned)

with open('results/tuned_metrics.txt', 'w') as f:
    f.write(f"Mean CV Macro F1: {tuned_f1:.4f}\n")
    f.write(f"Mean CV Macro Recall: {cv_results_tuned['test_recall_macro'].mean():.4f}\n")
    f.write(f"Mean CV Accuracy: {cv_results_tuned['test_accuracy'].mean():.4f}\n\n")
    f.write("Classification Report on Full Train Data (Tuned):\n")
    f.write(report_tuned)

print("=== MLP Performance Comparison ===")
print(f"Baseline Macro F1: {baseline_f1:.4f}")
print(f"Tuned Macro F1:    {tuned_f1:.4f}")
print(f"Improvement:       {tuned_f1 - baseline_f1:+.4f}")

# Save optimal model
joblib.dump(tuned_pipeline, 'mlp_best_model.pkl')
print("Optimal MLP Pipeline saved to mlp_best_model.pkl")


## Phase 4: Save Model

In [ ]:
if tuned_f1 > baseline_f1:
    print("Tuned model is better. Saving tuned model...")
    final_model = tuned_pipeline
else:
    print("Baseline model is better or equal. Saving baseline model...")
    final_model = base_pipeline

joblib.dump(final_model, 'mlp_best_model.pkl')
print("Model saved successfully to mlp_best_model.pkl")
